# CUDA Matrix Multiplication Performance Lab
Select **Runtime → Change runtime type → GPU** before running. This notebook uses the CUDA toolkit already present in the Colab runtime and stops with a clear error if `nvcc` is unavailable. Run cells from top to bottom after each session reset.

In [ ]:
REPO_URL = input('GitHub repository URL (https://github.com/OWNER/REPO): ').strip()
from urllib.parse import urlparse
parsed = urlparse(REPO_URL)
parts = parsed.path.strip('/').removesuffix('.git').split('/')
if parsed.scheme != 'https' or parsed.netloc != 'github.com' or len(parts) != 2 or not all(parts) or parsed.query or parsed.fragment:
    raise ValueError('Enter a public https://github.com/OWNER/REPO URL.')
print(f'Repository: {parts[0]}/{parts[1]}')

In [ ]:
import pathlib, shutil, subprocess
def run(*args, cwd=None):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No NVIDIA GPU utility found. In Colab, choose Runtime > Change runtime type > GPU and reconnect.')
run('nvidia-smi', '--query-gpu=name', '--format=csv,noheader')
for tool in ('git', 'cmake', 'nvcc'):
    if shutil.which(tool) is None:
        raise RuntimeError(f'{tool} is unavailable. Use a Colab GPU runtime with the CUDA toolkit; check !which nvcc and !nvcc --version.')
run('nvcc', '--version')
repo = pathlib.Path('/content/cuda-matmul-lab')
if repo.exists():
    origin = subprocess.check_output(['git', '-C', str(repo), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin.rstrip('/').removesuffix('.git') != REPO_URL.rstrip('/').removesuffix('.git'):
        raise RuntimeError(f'{repo} exists with a different origin: {origin}')
    run('git', 'pull', '--ff-only', cwd=repo)
else:
    run('git', 'clone', REPO_URL, repo)

In [ ]:
run('cmake', '-S', repo, '-B', repo / 'build', '-DCMAKE_BUILD_TYPE=Release')
run('cmake', '--build', repo / 'build', '-j2')

In [ ]:
binary = repo / 'build' / 'matmul_lab'
run(binary, '--test', cwd=repo)

In [ ]:
results_path = repo / 'docs' / 'results' / 'benchmark.json'
run(binary, '--benchmark', '--output', results_path, cwd=repo)

In [ ]:
import json
data = json.loads(results_path.read_text())
print('GPU:', data['environment']['gpu_name'])
print('Trials per kernel:', data['methodology']['trials'])
print(f"{'M×K · K×N':<22} {'Kernel':<8} {'Median ms':>10} {'Range ms':>18} {'GFLOP/s':>10} {'Correct':>8}")
for shape in data['results']:
    label = f"{shape['m']}×{shape['k']} · {shape['k']}×{shape['n']}"
    for kernel in shape['kernels']:
        timing = kernel['timing']
        span = f"{timing['min_ms']:.3f}–{timing['max_ms']:.3f}"
        print(f"{label:<22} {kernel['name']:<8} {timing['median_ms']:>10.3f} {span:>18} {kernel['gflops']:>10.2f} {str(kernel['correctness']['passed']):>8}")

In [ ]:
from google.colab import files
files.download(str(results_path))

## Optional longer sweep and profiling
To add three larger shapes, run `run(binary, '--benchmark', '--expanded', '--output', results_path, cwd=repo)` and download the new JSON. Profiling is optional: check `shutil.which('ncu')`, then follow the README only if Colab permits access to performance counters. Neither profiling nor the longer sweep is needed for the required run.